# NB Assist — Task 2, run on a Kaggle GPU

This notebook produces the **model-stack** numbers for the submission: the local multilingual retriever (`istiaqfuad/triBne-e5-small`, MIT) plus the local generator (`Qwen/Qwen3-1.7B`, Apache-2.0, 4-bit) answering the supplied `bank-faq.csv`, then the full 40-question evaluation.

**Read this before judging anything below**

- Nothing here reaches a customer. The questions come from the pack's `test-questions.csv`; the project contains no customer or company data.
- Model **weights** are downloaded from Hugging Face on first run, so internet must be on. Inference itself is local on the T4 — the offline system never calls an inference API.
- The 40 questions are **held out**. They are used only for measurement, never for training, tuning, or prompt selection.
- The embedder is a third-party fine-tune of `intfloat/multilingual-e5-small`, trained on CC-BY parallel pairs, so outside language data already reaches retrieval. Disclosed in `README.md`.
- Section 7 calls an **online** AI model. That is the brief's own comparison requirement, and it sends only those 40 synthetic sample questions.
- Section 6 is the brief's *"Extra (optional): train the small model further"*. Sections 1–5 and 7 are the mandatory part.

## 1. Install dependencies

In [ ]:
!pip -q install -r /kaggle/input/biai-bank-assistant/requirements-kaggle.txt

## 2. Copy the project to a writable directory

`/kaggle/input` is **read-only**, so `evaluate.py` cannot write `reports/offline_results.json` there. Copy the project into `/kaggle/working` and run everything from the copy.

In [ ]:
import os
import shutil

SRC = '/kaggle/input/biai-bank-assistant'   # the dataset you uploaded
DST = '/kaggle/working/nb_assist'           # writable working copy

shutil.rmtree(DST, ignore_errors=True)
for item in ('src', 'ui', 'data', 'evaluate.py', 'kaggle_demo.py', 'README.md', 'ONE_PAGE_NOTE.md'):
    src = os.path.join(SRC, item)
    dst = os.path.join(DST, item)
    if os.path.isdir(src):
        shutil.copytree(src, dst, ignore=shutil.ignore_patterns('__pycache__'))
    elif os.path.exists(src):
        shutil.copy2(src, dst)
os.makedirs(os.path.join(DST, 'reports'), exist_ok=True)

missing = [p for p in ('src/assistant.py', 'data/bank-faq.csv', 'data/test-questions.csv', 'evaluate.py')
           if not os.path.exists(os.path.join(DST, p))]
print('copied:', sorted(os.listdir(DST)))
assert not missing, f'upload is missing {missing} - re-upload the repository contents as a Kaggle dataset'

## 3. Build the assistant

Both models load eagerly here, and the assertions matter. If either one fails to load, the system silently degrades to lexical smoke-test mode and a Bangla question then gets an English verbatim-FAQ reply — which is exactly the failure that hid the scoring bug in the first draft of this submission.

In [ ]:
import sys

sys.path.insert(0, DST)
os.environ['BIAI_EMBEDDING_MODEL'] = 'istiaqfuad/triBne-e5-small'
os.environ['BIAI_GENERATION_MODEL'] = 'Qwen/Qwen3-1.7B'

from src.assistant import BankAssistant
from src.data import load_faq

assistant = BankAssistant(load_faq(os.path.join(DST, 'data', 'bank-faq.csv')))

print('retrieval mode     :', assistant.retriever.mode)
print('generator available:', assistant.generator.available)
print('generator status   :', assistant.generator.load_error or 'loaded')
assert assistant.retriever.dense_available, 'embedding model did not load'
assert assistant.generator.available, 'generator did not load'

import torch
print('device             :', assistant.generator.model.device)
print('quantised          :', getattr(assistant.generator.model, 'is_loaded_in_4bit', 'unknown'))
print('parameters         :', sum(p.numel() for p in assistant.generator.model.parameters())/1e9, 'B')

## 4. One question per script

The brief's requirements 1–4 in five calls: answers only from the FAQ, a reply in the language the customer used, everything on the small open model, and a hand-off for anything the FAQ cannot support.

In [ ]:
examples = [
    'Can I open a joint account with my spouse?',
    'ক্রেডিট কার্ডের বিল কবে দিতে হয়?',
    'Card er PIN bhule gechi, reset korbo kibhabe?',
    'Personal loan er interest rate koto?',
    "What is today's dollar to taka exchange rate?",
]

for question in examples:
    result = assistant.answer(question)
    print(f'Q   : {question}')
    print(f'A   : {result.answer}')
    print(f'      status={result.status}  source_id={result.source_id}  language={result.language}')
    print(f'      reason={result.support_type}  mode={result.metadata["mode"]}  '
          f'answer_form={result.metadata.get("answer_form", "-")}  '
          f'latency={result.latency_ms:.0f} ms\n')

## 5. The 40-question evaluation (mandatory)

`evaluate.py` runs the whole held-out set and writes `reports/offline_results.json` into the writable copy. Latency here is model-stack latency: retrieval plus two generator calls per question (an entailment check, then the answer).

In [ ]:
os.chdir(DST)
!python evaluate.py

In [ ]:
import json

report = json.load(open(os.path.join(DST, 'reports', 'offline_results.json'), encoding='utf-8'))
rows = report['rows']
answered = [r for r in rows if not r['expected_unknown']]
misses = [r for r in rows if not (r['source_correct'] and r['refusal_correct'])]
forms = {}
for r in answered:
    forms[r['answer_form']] = forms.get(r['answer_form'], 0) + 1
model_written_non_english = sum(
    1 for r in rows if r['status'] == 'answered' and r['language'] in {'bn', 'banglish'} and r['answer_form'] == 'model'
)

print(f"retrieval mode        {report['retrieval_mode']}")
print(f"generator loaded      {report['generator_loaded']}")
print(f"correct FAQ source    {sum(r['source_correct'] for r in answered)}/{len(answered)}")
print(f"correct hand-offs     {sum(r['refusal_correct'] for r in rows if r['expected_unknown'])}/{report['unknown']}")
print(f"overall gate accuracy {report['overall_gate_accuracy']:.1%}")
print(f"median / p95 latency  {report['median_latency_ms']:.0f} ms / {report['p95_latency_ms']:.0f} ms")
print(f"answer forms          {forms}")
print(f"non-English answers written by the model: {model_written_non_english}")
print('\nmisses:')
for r in misses:
    print(f"  id={r['id']:02d} got={r['source_id']} expected={r['expected_source_id']} "
          f"status={r['status']} reason={r['support_type']}  {r['question']}")
reasons = sorted({r['support_type'] for r in rows if r['support_type']})
print('\nhand-off reasons:', {reason: sum(1 for r in rows if r['support_type'] == reason) for reason in reasons})

## 6. Optional — train the small model further

The brief lists this as *"Extra (optional): train the small model further on banking words or Banglish, and show if it got better."* Sections 1–5 already stand on their own; skipping to section 7 loses nothing required.

Data: `istiaqfuad/bangla-english-banglish-pairs` (CC-BY-4.0, derived from `Helsinki-NLP/opus-100`). No FAQ text, no test questions and no customer data enter training. Two limits stated up front, because they are real:

- The `banglish` rows in that corpus pair Banglish with **Bangla**, not English, so they cannot supervise Banglish→English by themselves. Training here is banking-domain **Bangla→English**; Banglish queries stay on the hand-written glossary in `src/normalize.py`.
- Nothing is tuned against the 40 questions. One fixed configuration, two epochs, no checkpoint selection at all — the validation split is carved out of the corpus and left unused so that anyone checking the claim can see the 40 were never a selection signal.

In [ ]:
!pip -q install datasets peft

In [ ]:
import random
import re

from datasets import load_dataset

BANGLA = re.compile(r'[\u0980-\u09FF]')
BANK_TERMS = (
    'account', 'bank', 'loan', 'card', 'balance', 'interest', 'payment', 'pay ',
    'transfer', 'deposit', 'atm', 'branch', 'fee', 'charge', 'limit', 'withdraw',
    'credit', 'debit', 'money', 'taka', 'currency', 'statement', 'cheque', 'check',
    'pin', 'password', 'transaction', 'mobile banking', 'insurance', 'mortgage',
)

# Streaming: the full corpus is ~1 GB and 2.4M rows, and this step needs a few
# thousand banking-domain pairs.
stream = load_dataset('istiaqfuad/bangla-english-banglish-pairs', split='train', streaming=True)

pairs, seen, scanned = [], set(), 0
for row in stream:
    scanned += 1
    if len(pairs) >= 1200:
        break
    if row['source'] != 'en_bn':
        continue
    anchor, positive = row['anchor'] or '', row['positive'] or ''
    if bool(BANGLA.search(anchor)) == bool(BANGLA.search(positive)):
        continue                       # want exactly one Bangla side and one English side
    bangla = anchor if BANGLA.search(anchor) else positive
    english = positive if BANGLA.search(anchor) else anchor
    if not any(term in english.lower() for term in BANK_TERMS):
        continue
    if not (8 <= len(english) <= 160 and 6 <= len(bangla) <= 160):
        continue
    key = bangla.strip()
    if key in seen:
        continue
    seen.add(key)
    pairs.append((key, english.strip()))

random.seed(0)
random.shuffle(pairs)
cut = max(1, int(len(pairs) * 0.9))
train_pairs, val_pairs = pairs[:cut], pairs[cut:]
print(f'scanned {scanned} rows | training pairs {len(train_pairs)} | held-out validation pairs {len(val_pairs)}')
for bangla, english in train_pairs[:4]:
    print(f'  {bangla}  =>  {english}')

In [ ]:
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

INSTRUCTION = (
    "Translate the customer's Bangla banking query into plain English. "
    "Output only the English query.\nQuery: {bangla}\nEnglish: {english}\n"
)
MODEL_NAME = os.environ['BIAI_GENERATION_MODEL']
adapter_dir = '/kaggle/working/rewriter-lora'

# A separate 4-bit copy, so these adapters never touch the model that produced
# the section 5 results.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map='auto',
    dtype=torch.float16,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    ),
)
base.config.use_cache = False

model = get_peft_model(prepare_model_for_kbit_training(base), LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
))
model.print_trainable_parameters()

def encode(samples):
    texts = [INSTRUCTION.format(bangla=b, english=e) for b, e in samples['pairs']]
    out = tokenizer(texts, truncation=True, max_length=128, padding=False)
    out['labels'] = [ids[:] for ids in out['input_ids']]
    return out

training = Dataset.from_dict({'pairs': train_pairs}).map(encode, batched=True, remove_columns=['pairs'])
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=adapter_dir,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=True,
        optim='paged_adamw_8bit',
        logging_steps=25,
        save_strategy='no',
        report_to='none',
    ),
    train_dataset=training,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)
trainer.train()
model.save_pretrained(adapter_dir)
print('adapter saved to', adapter_dir)

In [ ]:
# Measure the shipped system, then the same system with a model-written English
# rewrite appended to the customer's own words. Only the 30 answerable questions
# are scored: the other 10 are meant to be refused, so they measure nothing here.
import csv

from evaluate import EXPECTED_SOURCE

QUESTIONS = {int(row['id']): row['question'] for row in csv.DictReader(
    open(os.path.join(DST, 'data', 'test-questions.csv'), encoding='utf-8-sig'))}


def rewrite(text: str) -> str:
    prompt = INSTRUCTION.format(bangla=text, english='')
    inputs = tokenizer([prompt], return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=32, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).split('\n')[0].strip()


before, after, shown = 0, 0, 0
for question_id, expected in sorted(EXPECTED_SOURCE.items()):
    question = QUESTIONS[question_id]
    got_before = assistant.retriever.search(question, top_k=1)[0].entry.faq_id
    english = rewrite(question)
    got_after = assistant.retriever.search(f'{question} {english}', top_k=1)[0].entry.faq_id
    before += got_before == expected
    after += got_after == expected
    if shown < 6:
        shown += 1
        print(f'id={question_id:02d} before={got_before:2d} after={got_after:2d} expected={expected:2d}  {question!r}')
        print(f'         rewrite -> {english!r}')

total = len(EXPECTED_SOURCE)
print(f'\ntop-1 FAQ correct before training: {before}/{total}')
print(f'top-1 FAQ correct after  training: {after}/{total}')
print('If that number did not rise, that is the honest result: the shipped submission keeps the')
print('un-fine-tuned model and the hand-written glossary, because the glossary is auditable.')

## 7. Mandatory — the same 40 questions on an online AI model, then compare

> *"Then run the same 40 questions on any online AI model and compare."*

This is the only section that talks to an outside service, and it sends **only** the 40 synthetic questions from the supplied pack. Add a key under **Kaggle → Notebooks → Settings → Add-ons → Secrets** named `GEMINI_API_KEY` (a free Google AI Studio key works) and keep internet enabled. Another hosted model can be used instead by setting `BIAI_BASELINE_MODEL`.

The comparison is kept fair, and where a machine cannot judge it, it is left visibly open:

- The online model gets **no FAQ**. It is asked as a generic assistant and told to answer with exactly `HANDOFF` when it does not know this bank's rule. That is the behaviour the brief wants compared.
- Correct hand-off rate on the 10 uncovered questions and latency are computed automatically.
- Invented figures are computed automatically: any number in an online answer that appears nowhere in the FAQ.
- Whether its **content** is right on the 30 answerable questions needs a human read. All 40 answers are saved to `reports/baseline_results.csv` for that, and the table below says `needs human review` rather than inventing a score.

In [ ]:
BASELINE_MODEL = os.environ.get('BIAI_BASELINE_MODEL', 'gemini-2.0-flash')


def api_key():
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('GEMINI_API_KEY')
        if key:
            return key
    except Exception:
        pass
    return os.environ.get('GEMINI_API_KEY') or None


client, key = None, api_key()
if client is None and key:
    from google import genai
    client = genai.Client(api_key=key)
print('baseline model:', BASELINE_MODEL, '| key found:', bool(key))
if client is None:
    print('no key: section 7 will print the manual fallback instead of failing')

In [ ]:
import csv
import time

from evaluate import UNKNOWN_IDS
from src.data import load_faq
from src.normalize import detect_language
from src.retrieval import extract_numbers

FAQ_NUMBERS = set()
for entry in load_faq(os.path.join(DST, 'data', 'bank-faq.csv')):
    FAQ_NUMBERS |= extract_numbers(entry.searchable_text)

PROMPT = (
    "You are a bank chat assistant. Answer the customer directly in the language they wrote in. "
    "If you do not actually know this bank's specific rule, reply with exactly HANDOFF and nothing "
    "else.\n\nCustomer: {question}\n\nAnswer:"
)
baseline_path = os.path.join(DST, 'reports', 'baseline_results.csv')

if client is None:
    print('Online run skipped. Two ways to still complete this mandatory comparison:')
    print('  1. add the GEMINI_API_KEY secret, enable internet, re-run the two cells above; or')
    print('  2. paste the 40 questions into any hosted chat model, save its 40 replies in the same')
    print(f'     order in a CSV with the columns used below, and put that file at {baseline_path}.')
else:
    rows = []
    for row in csv.DictReader(open(os.path.join(DST, 'data', 'test-questions.csv'), encoding='utf-8-sig')):
        question_id, question = int(row['id']), row['question']
        started = time.perf_counter()
        try:
            answer = (client.models.generate_content(model=BASELINE_MODEL,
                                                     contents=PROMPT.format(question=question)).text or '').strip()
        except Exception as exc:
            answer = f'ERROR {type(exc).__name__}: {exc}'
        elapsed = (time.perf_counter() - started) * 1000
        handed_off = answer.upper().startswith('HANDOFF')
        rows.append({
            'id': question_id,
            'question': question,
            'answer': answer,
            'expected_unknown': question_id in UNKNOWN_IDS,
            'handed_off': handed_off,
            'refusal_correct': handed_off == (question_id in UNKNOWN_IDS),
            'language': detect_language(question),
            'invented_figures': sorted(extract_numbers(answer) - FAQ_NUMBERS) if not handed_off else [],
            'latency_ms': elapsed,
        })
        print(f'{question_id:02d} handoff={handed_off!s:5} {elapsed:7.0f} ms  {answer[:90]!r}')
    with open(baseline_path, 'w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print('\nsaved', baseline_path)

In [ ]:
import ast
import csv
import json
import os
import statistics

from evaluate import UNKNOWN_IDS
from src.data import load_faq
from src.retrieval import extract_numbers

BASELINE_MODEL = os.environ.get('BIAI_BASELINE_MODEL', 'gemini-2.0-flash')
FAQ_NUMBERS = set()
for entry in load_faq(os.path.join(DST, 'data', 'bank-faq.csv')):
    FAQ_NUMBERS |= extract_numbers(entry.searchable_text)

path = os.path.join(DST, 'reports', 'baseline_results.csv')
rows = list(csv.DictReader(open(path, encoding='utf-8'))) if os.path.exists(path) else []
if not rows:
    print('nothing to compare yet: run the cell above, or place a manual CSV at', path)
else:
    def as_bool(value):
        return str(value).strip().lower() in {'1', 'true', 'yes'}

    answered = [r for r in rows if not as_bool(r['expected_unknown'])]
    uncovered = [r for r in rows if as_bool(r['expected_unknown'])]
    online_refusals = sum(as_bool(r['handed_off']) for r in uncovered)
    def figure_list(value):
        try:
            parsed = ast.literal_eval(value or '[]')
        except (SyntaxError, ValueError):
            parsed = []
        return parsed if isinstance(parsed, list) else []

    online_invented = sum(bool(figure_list(r['invented_figures'])) for r in answered)
    online_latency = [float(r['latency_ms']) for r in rows if str(r.get('latency_ms', '')).replace('.', '', 1).isdigit()]

    offline = json.load(open(os.path.join(DST, 'reports', 'offline_results.json'), encoding='utf-8'))
    offline_rows = offline['rows']
    offline_answered = [r for r in offline_rows if not r['expected_unknown']]
    offline_invented = sum(bool(extract_numbers(r['answer']) - FAQ_NUMBERS) for r in offline_answered)

    print('| | NB Assist — local Qwen3-1.7B, offline | Online AI model |')
    print('|---|---|---|')
    print(f'| model | {os.environ["BIAI_GENERATION_MODEL"]} (4-bit, on a T4) | {BASELINE_MODEL} (hosted) |')
    print(f'| customer data leaves the machine | no | yes (40 sample questions only) |')
    print(f'| correct hand-offs on the 10 uncovered | '
          f'{sum(r["refusal_correct"] for r in offline_rows if r["expected_unknown"])}/{len([r for r in offline_rows if r["expected_unknown"]])} '
          f'| {online_refusals}/{len(uncovered)} |')
    print(f'| correct FAQ source on the 30 answerable | '
          f'{sum(r["source_correct"] for r in offline_answered)}/{len(offline_answered)} '
          f'| needs human review — answers are in {os.path.basename(path)} |')
    print(f'| answers containing a figure absent from the FAQ | {offline_invented}/{len(offline_answered)} (blocked by the grounding gate) '
          f'| {online_invented}/{len(answered)} |')
    if online_latency:
        print(f'| median latency | {offline["median_latency_ms"]:.0f} ms | {statistics.median(online_latency):.0f} ms (network included) |')
    print()
    print('The defensible claim is that the offline system cannot print an unsourced number, not that')
    print('its prose is better than a hosted model. Read the CSV before writing the sentence.')

## 8. Submission checklist

| Deliverable | Where it lives | Weight (brief's own table) |
|---|---|---|
| Loom video, ≤5 minutes | record it from section 4 and section 5 | 20% |
| Works on the supplied sample files | `reports/offline_results.json` | 25% |
| Clear build that handles mistakes | `src/assistant.py` gate order + the hand-off reasons printed above | 20% |
| Hardest part solved | cross-script retrieval + the groundedness guards | 15% |
| Clean code runnable from the README | `README.md` | 10% |
| Honesty in the note | `ONE_PAGE_NOTE.md`, "What does not work well" | 10% |

After the run: paste section 5's numbers into the **Model stack** column of `ONE_PAGE_NOTE.md` and section 7's table into its comparison paragraph. Both are still `pending run` until you do.